# 08 — Asset query

Everything the risk model says about one stock:

1. **Profile**: predicted total, factor and specific vol, predicted beta, and
   realized vol and beta for comparison
2. **Factor exposures**, with where each sits in the universe
3. **Risk decomposition**: which factors the stock's variance comes from
4. **Covariance neighbours**: the names it co-moves with most and least
5. **History**: predicted vol against what happened, and exposures through time

Set `QUERY` to a ticker, permno or company-name fragment and run top to bottom.
The **dropdown** at the end does the same interactively when the notebook is
open in Jupyter. `pair(a, b)` explains the covariance between any two names.

All figures are annualized, and the model is the one at `AS_OF` (default:
latest). Covariances come from one row of Σ, `X @ (F @ x_i)`; the full matrix is
never built.

In [ ]:
import os
import sys

sys.path.insert(0, os.getcwd())

%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

import db
import query
import riskmodel

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

INK, ACCENT, MUTED, GREEN, AMBER = "#2a78d6", "#e34948", "#8a8a83", "#1baf7a", "#eda100"
STYLE_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
GROUP_COLORS = {"market": INK, "industry": AMBER, "style": GREEN, "specific": MUTED}


def tidy(ax):
    ax.grid(alpha=.25, lw=.6)
    ax.set_axisbelow(True)
    for s in ("top", "right"):
        ax.spines[s].set_visible(False)


def label_ends(ax, ends, min_gap, fontsize=8):
    """Direct labels at line ends, nudged apart so close finishers stay legible.

    ``ends`` maps label -> (x, y, color).
    """
    last = -np.inf
    for name, (x, y, c) in sorted(ends.items(), key=lambda kv: kv[1][1]):
        y = max(y, last + min_gap)
        last = y
        ax.annotate(f" {name}", (x, y), color=c, fontsize=fontsize, va="center")


# ---- inputs -----------------------------------------------------------------
QUERY = "CVX"        # ticker, permno, or part of the company name ("chevron")
AS_OF = None         # forecast date; None = latest
TOP_N = 10           # neighbours to list

In [ ]:
engine = db.get_engine()
db.check_connection(engine)
store = riskmodel.ModelStore.from_db(engine)
model = store.at(AS_OF)
print(f"model date {model.date.date()}: {len(model.ids)} names priced")

---
## The menu

Every name the model prices at this date. Only these can be queried: the model
covers the 220-name sample, and ~117 of them are still listed at the latest date.

In [ ]:
menu = query.universe(model)
print(menu.groupby("sector_name").size().rename("names").to_frame().T.to_string())
display(menu.style.format({"cap": "{:,.0f}", "total_vol": "{:.1%}", "specific_vol": "{:.1%}", "beta": "{:.2f}"}))

---
## The report

In [ ]:
def fmt_profile(p):
    out = p.copy().astype(object)
    for k in out.index:
        v = out[k]
        if isinstance(v, (float, np.floating)) and np.isfinite(v):
            if "vol" in k or "%" in k:
                out[k] = f"{v:.1%}"
            elif "beta" in k:
                out[k] = f"{v:.2f}"
            elif "cap" in k:
                out[k] = f"{v:,.0f}"
            elif k == "id":
                out[k] = f"{v:.0f}"
    return out.to_frame()


def show(q, top_n=TOP_N):
    r = query.asset_report(store, model, q, top_n)
    p = r["profile"]
    display(Markdown(f"### {p['ticker']} — {p['company']}  \n{p['sector']}, forecast date {p['forecast date']}"))
    display(fmt_profile(p))

    # -- exposures and risk decomposition ------------------------------------
    ex = r["exposures"]
    styles = list(ex.index)
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [1.3, 1]})
    # Universe distribution per style as a strip, the asset as a marker.
    for i, s in enumerate(styles):
        ax[0].scatter(model.X[s], np.full(len(model.ids), i) + np.random.default_rng(i).uniform(-.18, .18, len(model.ids)),
                      s=6, color=MUTED, alpha=.35, lw=0)
        ax[0].scatter(ex.loc[s, "exposure"], i, s=90, color=ACCENT, zorder=5, edgecolor="white", lw=1.2)
        ax[0].annotate(f"  {ex.loc[s, 'exposure']:+.2f}  (p{ex.loc[s, 'universe percentile'] * 100:.0f})",
                       (ex.loc[s, "exposure"], i), fontsize=8, color=ACCENT, va="bottom")
    ax[0].set_yticks(range(len(styles)), styles)
    ax[0].axvline(0, color="#1a1a19", lw=.8)
    ax[0].set(title="Style exposures vs the universe (grey)", xlabel="exposure, σ units")

    rc = r["risk_contrib"].copy()
    rc["group"] = [("specific" if f == "specific" else riskmodel._group(f)) for f in rc.index]
    grp = rc.groupby("group")["pct_of_total_var"].sum().reindex(GROUP_COLORS.keys()).fillna(0)
    ax[1].barh(range(len(grp)), grp * 100, color=[GROUP_COLORS[g] for g in grp.index])
    for i, v in enumerate(grp):
        ax[1].annotate(f" {v:.0%}", (max(v, 0) * 100, i), va="center", fontsize=9)
    ax[1].set_yticks(range(len(grp)), grp.index)
    ax[1].invert_yaxis()
    ax[1].set(title="Where its variance comes from", xlabel="% of variance")
    for a in ax:
        tidy(a)
    plt.tight_layout()
    plt.show()
    display(rc.drop(columns="group").style.format(
        {"exposure": "{:+.3f}", "var_contrib": "{:.4f}", "pct_of_total_var": "{:.1%}"}, na_rep=""))

    # -- neighbours -----------------------------------------------------------
    peers = r["all_peers"]
    fmt = {"cov": "{:.4f}", "corr": "{:.2f}"}
    display(Markdown(f"**Highest correlation** (top {top_n})"))
    display(r["highest_corr"].style.format(fmt))
    display(Markdown(f"**Lowest correlation** (bottom {top_n}), the best diversifiers against it"))
    display(r["lowest_corr"].style.format(fmt))
    display(Markdown("**Highest / lowest covariance.** Covariance also scales with the other name's vol, "
                     "so this list favours volatile names that correlation would not."))
    display(pd.concat({"highest": r["highest_cov"][["ticker", "sector_name", "cov", "corr"]].reset_index(drop=True),
                       "lowest": r["lowest_cov"][["ticker", "sector_name", "cov", "corr"]].reset_index(drop=True)},
                      axis=1).style.format({("highest", "cov"): "{:.4f}", ("lowest", "cov"): "{:.4f}",
                                            ("highest", "corr"): "{:.2f}", ("lowest", "corr"): "{:.2f}"}))

    fig, ax = plt.subplots(figsize=(13, 2.8))
    same = peers["same_sector"]
    ax.hist(peers.loc[~same, "corr"], bins=40, color=MUTED, alpha=.8, label="other sectors")
    ax.hist(peers.loc[same, "corr"], bins=20, color=INK, alpha=.9, label=f"same sector ({p['sector']})")
    ax.set(title=f"Predicted correlation of {p['ticker']} with every other name", xlabel="correlation",
           ylabel="names")
    ax.legend(frameon=False, fontsize=9)
    tidy(ax)
    plt.tight_layout()
    plt.show()

    # -- history --------------------------------------------------------------
    h = r["history"]
    ri = r["realized"]
    fwd = ri.rolling(12).std().shift(-11) * np.sqrt(12)
    fig, ax = plt.subplots(1, 2, figsize=(13, 3.6))
    ax[0].plot(h.index, h["total_vol"] * 100, color=INK, lw=2)
    ax[0].plot(h.index, h["specific_vol"] * 100, color=INK, lw=1, ls="--")
    ends = {"predicted": (h.index[-1], h["total_vol"].iloc[-1] * 100, INK),
            "specific": (h.index[-1], h["specific_vol"].iloc[-1] * 100, INK)}
    fw = fwd.reindex(h.index).dropna()
    if len(fw):
        ax[0].plot(fw.index, fw * 100, color=ACCENT, lw=2)
        ends["realized, next 12m"] = (fw.index[-1], fw.iloc[-1] * 100, ACCENT)
    lo, hi = ax[0].get_ylim()
    label_ends(ax[0], ends, (hi - lo) * .06, fontsize=9)
    ax[0].set(title="Predicted vol at t vs realized over the next 12 months", ylabel="%/yr")
    ax[0].margins(x=.2)
    for s, c in zip(styles, STYLE_COLORS):
        ax[1].plot(h.index, h[s], color=c, lw=1.6)
    lo, hi = ax[1].get_ylim()
    label_ends(ax[1], {s: (h.index[-1], h[s].iloc[-1], c) for s, c in zip(styles, STYLE_COLORS)},
               (hi - lo) * .055)
    ax[1].axhline(0, color="#1a1a19", lw=.8)
    ax[1].set(title="Style exposures through time", ylabel="σ units")
    ax[1].margins(x=.2)
    for a in ax:
        tidy(a)
    plt.tight_layout()
    plt.show()
    return r


report = show(QUERY)

---
## Why two names co-move

The covariance between two different names is purely factor-driven (the model
sets specific cross-terms to zero), so it splits exactly into per-factor pieces.
Here: the queried name against its most-correlated neighbour.

In [ ]:
def pair(a, b):
    pr = query.pair_report(store, model, a, b)
    s = pr["summary"].copy()
    for k in ("predicted cov (ann.)",):
        s[k] = f"{s[k]:.4f}"
    for k in ("predicted corr", "realized corr, trailing 60m"):
        s[k] = f"{s[k]:.2f}" if pd.notna(s[k]) else "n/a"
    display(s.to_frame())
    display(pr["by_factor"].style.format({"exposure a": "{:+.3f}", "exposure b": "{:+.3f}", "cov contrib": "{:+.4f}"}))
    return pr


_ = pair(QUERY, report["highest_corr"]["ticker"].iloc[0])

---
## Interactive

Pick a name and the report re-renders below. This needs a live kernel; in a
statically rendered copy of the notebook the dropdown doesn't appear.

In [ ]:
try:
    import ipywidgets as widgets

    opts = [(f"{r.ticker} — {r.conm} ({r.sector_name})", idx) for idx, r in menu.sort_values("ticker").iterrows()]
    dd = widgets.Dropdown(options=opts, value=report["id"], description="asset", layout=widgets.Layout(width="520px"))
    out = widgets.Output()

    def _on_change(change):
        if change["name"] == "value":
            out.clear_output(wait=True)
            with out:
                show(change["new"])

    dd.observe(_on_change, names="value")
    display(dd, out)
except ImportError:
    print("ipywidgets not available — set QUERY above and re-run instead.")